In [97]:
import torch
from torch import nn
import torch.nn.functional as F

# 1 Load the data
with open("shakes.txt", "r") as f:
    text = f.read()
    
# 2 preporcess the data
## create preprocesor - tokenizer
chars = sorted(list(set(text)))
vocab_size = len(chars)

## int : str mapping
itos = {i:st for i,st in enumerate(chars)}
stoi = {st:i for i,st in enumerate(chars)}

# encode decode
encode = lambda inp : [stoi[i] for i in inp]
decode = lambda inp : "".join([itos[i] for i in inp])

data = torch.tensor(encode(text))
len(data)

1115394

In [98]:
## split the data
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]
len(train_data),  len(val_data)


batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == "train" else val_data
    idx = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in idx])
    y = torch.stack([data[i+1:i+block_size+1] for i in idx])
    return x, y

xb, yb = get_batch("train")

xb.shape, yb.shape

(torch.Size([4, 8]), torch.Size([4, 8]))

In [137]:
### 3. create model
torch.manual_seed(24)
class BiagramModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        ## This will convetr (B,T) into (B,T,65(vocab_size))--> B,T,C
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size) ## 
        
    def forward(self, inpx, target= None):
        
        logits = self.token_embedding_table(inpx) ## B,T ---> BTC
        B,T,C = logits.shape
        if target is None:
            loss = None
        else:
            logits = logits.view(B*T,C)
            target = target.view(B*T)
            loss = F.cross_entropy(logits, target)
            
        return logits, loss
    
    def generate(self, idx, max_length):
        for i in range(max_length):
            logit, _= self(idx)
            logit = logit[:,-1,:]
            probs =  torch.softmax(logit, dim=-1)
            print(probs)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx,idx_next), dim=1)
        return idx
        
            
        
    

model = BiagramModel(vocab_size=vocab_size)
logit, loss = model(xb,yb)
print(loss)


        

tensor(4.8104, grad_fn=<NllLossBackward0>)


In [131]:
### Training 
max_epoch = 1000
learning_rate = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
for epoch in range(max_epoch):
    xb, yb = get_batch("train")
    _, loss = model(xb,yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"The loss is {loss.item()}")

    
    


The loss is 2.500797748565674
The loss is 2.749443769454956
The loss is 2.710019111633301
The loss is 2.981529474258423
The loss is 2.4890379905700684
The loss is 2.3579375743865967
The loss is 2.664912700653076
The loss is 3.0634925365448
The loss is 2.6315205097198486
The loss is 2.627185821533203
The loss is 2.2901201248168945
The loss is 2.4623565673828125
The loss is 2.618834972381592
The loss is 2.6891090869903564
The loss is 2.7682507038116455
The loss is 2.6501121520996094
The loss is 2.909207582473755
The loss is 2.73679518699646
The loss is 2.5363450050354004
The loss is 2.9269235134124756
The loss is 2.963157892227173
The loss is 3.0173730850219727
The loss is 2.7786548137664795
The loss is 2.9064249992370605
The loss is 3.052851676940918
The loss is 3.1165220737457275
The loss is 2.9579179286956787
The loss is 2.757420301437378
The loss is 2.8869407176971436
The loss is 2.945096969604492
The loss is 3.032282829284668
The loss is 2.4484634399414062
The loss is 1.978199124336

In [138]:
test_inp = torch.zeros(size=(1,1), dtype=torch.int64)
print(decode(model.generate(test_inp, max_length=70).tolist()[0]))

tensor([[0.0019, 0.0063, 0.0065, 0.0026, 0.0784, 0.0054, 0.0220, 0.0017, 0.0183,
         0.0220, 0.0113, 0.0078, 0.0373, 0.0168, 0.0082, 0.0332, 0.0027, 0.0206,
         0.0041, 0.0074, 0.0086, 0.0135, 0.0047, 0.0067, 0.0010, 0.0076, 0.0124,
         0.0036, 0.0064, 0.0150, 0.0481, 0.0137, 0.0092, 0.0123, 0.0034, 0.1236,
         0.0014, 0.0227, 0.0246, 0.0054, 0.0005, 0.0081, 0.0410, 0.0067, 0.0032,
         0.0319, 0.0309, 0.0032, 0.0048, 0.0062, 0.0061, 0.0157, 0.0045, 0.0141,
         0.0048, 0.0063, 0.0210, 0.0184, 0.0455, 0.0056, 0.0128, 0.0315, 0.0039,
         0.0073, 0.0073]], grad_fn=<SoftmaxBackward0>)




In [134]:
model.generate(test_inp, max_length=70)

tensor([[ 0, 32, 53,  1, 49, 52, 42,  1, 46, 53, 58, 53, 51, 63,  1, 45, 56,  1,
         44, 35, 50,  8,  0,  0,  0, 15, 21, 24, 43, 58, 47, 50, 53, 44, 62, 36,
         35, 40, 60, 43, 58,  1, 58, 53, 56, 47, 51, 44, 43, 56, 58, 50,  1, 57,
          8,  0, 21, 57, 43,  0, 32, 47, 43,  1, 53, 52, 43, 56,  1, 46,  1]])

In [136]:
test_inp

tensor([[0]])